# iFood Case - Analysis


1. **Pergunta 1**: Qual a média de valor total (total\_amount) recebido em um mês considerando todos os yellow táxis da frota?
2. **Pergunta 2**: Qual a média de passageiros (passenger\_count) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?

---

## Pergunta 1:

Qual a média de valor total (total_amount) recebido em um mês considerando todos os yellow táxis da frota?

In [0]:
from pyspark.sql.functions import month, avg

avg_mensal = spark.sql("""
    select month(dh_inicio_corrida) as mes,
    avg(vl_total_corrida) avg_mensal_corridas
    from gold.taxidata.yellow_taxi_data
    group by all
    order by month(dh_inicio_corrida)
    """)
display(avg_mensal)

Databricks visualization. Run in Databricks to view.

## Pergunta 2:

Qual a média de passageiros (passenger_count) por cada hora do dia que pegaram táxi no mês de maio considerando todos os táxis da frota?

In [0]:
total_runs = spark.table("gold.taxidata.yellow_taxi_data").unionByName(spark.table("gold.taxidata.green_taxi_data"))

avg_passageiros = spark.sql("""
    select hour(dh_inicio_corrida) hora,
    round(avg(nr_passageiros),2) avg_passageiros
    from {total_runs}
    where date(dh_inicio_corrida) between date '2023-05-01' and date '2023-05-31' 
    group by all
    order by hour(dh_inicio_corrida)""",total_runs = total_runs)

display(avg_passageiros)

Databricks visualization. Run in Databricks to view.

---

## 🔍 Análises Adicionais

Explorando padrões nos dados de táxi de NYC

### Análise 3: Comparação Yellow vs Green Taxi

Comparando métricas principais entre os dois tipos de táxi da frota de NYC

In [0]:
# Comparação Yellow vs Green Taxi
comparacao = spark.sql("""
    SELECT 
        'Yellow Taxi' as tipo,
        COUNT(*) as total_corridas,
        ROUND(AVG(vl_total_corrida), 2) as ticket_medio,
        ROUND(AVG(nr_passageiros), 2) as passageiros_medio,
        ROUND(SUM(vl_total_corrida) / 1000000, 2) as receita_total_milhoes
    FROM gold.taxidata.yellow_taxi_data
    
    UNION ALL
    
    SELECT 
        'Green Taxi' as tipo,
        COUNT(*) as total_corridas,
        ROUND(AVG(vl_total_corrida), 2) as ticket_medio,
        ROUND(AVG(nr_passageiros), 2) as passageiros_medio,
        ROUND(SUM(vl_total_corrida) / 1000000, 2) as receita_total_milhoes
    FROM gold.taxidata.green_taxi_data
""")

display(comparacao)

Databricks visualization. Run in Databricks to view.

### Análise 4: Distribuição de Receita e Volume por Mês

Entendendo a relação entre volume de corridas e receita média ao longo dos meses

In [0]:
# Distribuição de volume e receita por mês
volume_receita = spark.sql("""
    SELECT 
        month(dh_inicio_corrida) as mes,
        ROUND(COUNT(*) / 1000000.0, 2) as volume_milhoes_corridas,
        ROUND(AVG(vl_total_corrida), 2) as ticket_medio,
        ROUND(SUM(vl_total_corrida) / 1000000, 2) as receita_milhoes
    FROM gold.taxidata.yellow_taxi_data
    GROUP BY month(dh_inicio_corrida)
    ORDER BY mes
""")

display(volume_receita)

Databricks visualization. Run in Databricks to view.

### Análise 5: Padrões de Horário de Pico

Identificando os horários mais lucrativos e com maior demanda em Maio 2023

In [0]:
# Análise detalhada de horários de pico
peak_hours = spark.sql("""
    SELECT 
        hour(dh_inicio_corrida) as hora,
        ROUND(COUNT(*) / 1000.0, 1) as volume_mil_corridas,
        ROUND(AVG(vl_total_corrida), 2) as ticket_medio,
        ROUND(AVG(nr_passageiros), 2) as passageiros_medio,
        ROUND(SUM(vl_total_corrida) / 1000000, 2) as receita_milhoes
    FROM gold.taxidata.yellow_taxi_data
    WHERE month(dh_inicio_corrida) = 5
    GROUP BY hour(dh_inicio_corrida)
    ORDER BY hora
""")

display(peak_hours)

Databricks visualization. Run in Databricks to view.